# DepegScope: Data Exploration

This notebook explores the collected data for stablecoins, protocols, and their exposures.

## Contents
1. Setup and Data Loading
2. Stablecoin Analysis
3. Protocol Analysis
4. Exposure Analysis
5. Data Quality Checks

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import RAW_DATA_DIR, PROCESSED_DATA_DIR

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

## 1. Load Data

In [ ]:
# Load stablecoins
stablecoins_file = PROCESSED_DATA_DIR / "stablecoins_processed.json"
if stablecoins_file.exists():
    with open(stablecoins_file) as f:
        stablecoins_data = json.load(f)
    stablecoins_df = pd.DataFrame(stablecoins_data)
    print(f"Loaded {len(stablecoins_df)} stablecoins")
else:
    print(f"File not found: {stablecoins_file}")
    print("Run: python scripts/collect_data.py && python scripts/build_graph.py")

In [ ]:
# Load protocols
protocols_file = PROCESSED_DATA_DIR / "protocols_processed.json"
if protocols_file.exists():
    with open(protocols_file) as f:
        protocols_data = json.load(f)
    protocols_df = pd.DataFrame(protocols_data)
    print(f"Loaded {len(protocols_df)} protocols")
else:
    print(f"File not found: {protocols_file}")

In [ ]:
# Load exposures
exposures_file = PROCESSED_DATA_DIR / "exposures_processed.json"
if exposures_file.exists():
    with open(exposures_file) as f:
        exposures_data = json.load(f)
    exposures_df = pd.DataFrame(exposures_data)
    print(f"Loaded {len(exposures_df)} exposure relationships")
else:
    print(f"File not found: {exposures_file}")

## 2. Stablecoin Analysis

In [ ]:
# Display stablecoin summary
if 'stablecoins_df' in dir():
    display(stablecoins_df.head(10))
    print(f"\nColumns: {stablecoins_df.columns.tolist()}")

In [ ]:
# Market cap distribution
if 'stablecoins_df' in dir() and 'market_cap' in stablecoins_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Top 10 by market cap
    top_10 = stablecoins_df.nlargest(10, 'market_cap')
    axes[0].barh(top_10['symbol'], top_10['market_cap'] / 1e9)
    axes[0].set_xlabel('Market Cap (Billions USD)')
    axes[0].set_title('Top 10 Stablecoins by Market Cap')
    axes[0].invert_yaxis()
    
    # Market cap distribution (log scale)
    axes[1].hist(np.log10(stablecoins_df['market_cap'] + 1), bins=30, edgecolor='black')
    axes[1].set_xlabel('Log10(Market Cap)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Market Cap Distribution (Log Scale)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Stablecoin type distribution
if 'stablecoins_df' in dir() and 'stablecoin_type' in stablecoins_df.columns:
    type_counts = stablecoins_df['stablecoin_type'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    type_counts.plot(kind='pie', autopct='%1.1f%%', ax=ax)
    ax.set_title('Stablecoin Types Distribution')
    ax.set_ylabel('')
    plt.show()

## 3. Protocol Analysis

In [ ]:
# Display protocol summary
if 'protocols_df' in dir():
    display(protocols_df.head(10))
    print(f"\nColumns: {protocols_df.columns.tolist()}")

In [ ]:
# TVL distribution
if 'protocols_df' in dir() and 'tvl' in protocols_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Top 20 by TVL
    top_20 = protocols_df.nlargest(20, 'tvl')
    axes[0].barh(top_20['name'], top_20['tvl'] / 1e9)
    axes[0].set_xlabel('TVL (Billions USD)')
    axes[0].set_title('Top 20 Protocols by TVL')
    axes[0].invert_yaxis()
    
    # TVL distribution (log scale)
    axes[1].hist(np.log10(protocols_df['tvl'] + 1), bins=30, edgecolor='black')
    axes[1].set_xlabel('Log10(TVL)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('TVL Distribution (Log Scale)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Protocol category distribution
if 'protocols_df' in dir() and 'category' in protocols_df.columns:
    category_counts = protocols_df['category'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    category_counts.plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.set_title('Protocol Categories')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Exposure Analysis

In [ ]:
# Display exposure summary
if 'exposures_df' in dir():
    display(exposures_df.head(10))
    print(f"\nColumns: {exposures_df.columns.tolist()}")
    print(f"\nTotal exposure: ${exposures_df['amount_usd'].sum():,.0f}")

In [ ]:
# Exposure by stablecoin
if 'exposures_df' in dir():
    exposure_by_stable = exposures_df.groupby('stablecoin_symbol')['amount_usd'].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    exposure_by_stable.head(15).plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_xlabel('Stablecoin')
    ax.set_ylabel('Total Exposure (USD)')
    ax.set_title('Protocol Exposure by Stablecoin')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Exposure heatmap (top protocols x top stablecoins)
if 'exposures_df' in dir():
    # Create pivot table
    pivot = exposures_df.pivot_table(
        index='protocol_name',
        columns='stablecoin_symbol',
        values='amount_usd',
        aggfunc='sum',
        fill_value=0
    )
    
    # Get top 15 protocols and top 8 stablecoins
    top_protocols = exposures_df.groupby('protocol_name')['amount_usd'].sum().nlargest(15).index
    top_stables = exposures_df.groupby('stablecoin_symbol')['amount_usd'].sum().nlargest(8).index
    
    heatmap_data = pivot.loc[pivot.index.isin(top_protocols), pivot.columns.isin(top_stables)]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        heatmap_data / 1e9,
        annot=True,
        fmt='.1f',
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Exposure (Billions USD)'}
    )
    ax.set_title('Protocol-Stablecoin Exposure Matrix')
    plt.tight_layout()
    plt.show()

## 5. Data Quality Checks

In [ ]:
# Check for missing data
print("=== Missing Data Analysis ===")

if 'stablecoins_df' in dir():
    print(f"\nStablecoins missing values:")
    print(stablecoins_df.isnull().sum())

if 'protocols_df' in dir():
    print(f"\nProtocols missing values:")
    print(protocols_df.isnull().sum())

if 'exposures_df' in dir():
    print(f"\nExposures missing values:")
    print(exposures_df.isnull().sum())

In [ ]:
# Coverage analysis
print("=== Coverage Analysis ===")

if 'exposures_df' in dir() and 'protocols_df' in dir() and 'stablecoins_df' in dir():
    unique_protocols = exposures_df['protocol_name'].nunique()
    unique_stables = exposures_df['stablecoin_symbol'].nunique()
    
    print(f"\nProtocols with exposure data: {unique_protocols} / {len(protocols_df)}")
    print(f"Stablecoins with exposure data: {unique_stables} / {len(stablecoins_df)}")
    
    # Check for protocols without any exposure data
    protocols_with_exposure = set(exposures_df['protocol_name'].unique())
    if 'slug' in protocols_df.columns:
        all_protocols = set(protocols_df['slug'].unique())
        missing_protocols = all_protocols - protocols_with_exposure
        print(f"\nProtocols without exposure data: {len(missing_protocols)}")

In [ ]:
# Summary statistics
print("=== Summary Statistics ===")

if 'stablecoins_df' in dir():
    total_market_cap = stablecoins_df['market_cap'].sum()
    print(f"\nTotal stablecoin market cap: ${total_market_cap:,.0f}")

if 'protocols_df' in dir():
    total_tvl = protocols_df['tvl'].sum()
    print(f"Total protocol TVL: ${total_tvl:,.0f}")

if 'exposures_df' in dir():
    total_exposure = exposures_df['amount_usd'].sum()
    print(f"Total mapped exposure: ${total_exposure:,.0f}")
    if 'protocols_df' in dir():
        print(f"Exposure coverage: {total_exposure/total_tvl*100:.1f}% of TVL")

## Next Steps

Continue to the next notebook for network analysis:
- `02_network_analysis.ipynb`